# HarborGuard LoRA 训练（C4 · Colab · Unsloth + TRL SFT）

任务：把 C3 的「上下文 JSON → 建议 JSON」窄任务蒸馏到本地小模型，导出 **配对 GGUF**（未微调基座 + 微调后模型）交给 Ollama 对比评估。

**运行前**：上传 `ml/lora/datasets/train.jsonl`（由 `python -m ml.lora.distill` 生成，501 条）。
**本笔记本不读取 test.jsonl**——测试集只在本机部署后由 `python -m ml.lora.eval` 使用，
不参与训练、提示词修改或检查点选择；笔记本内的冒烟生成用训练集尾部留出的验证切片。

训练口径：
- 基座：Qwen3.5 小尺寸（先 0.8B，模型名以 Unsloth 文档为准）。**只在「模型不存在」时**
  按候选链降级，每次降级打印醒目警告并写入 `training_meta.json`；其他异常直接终止，
  不静默换模型。配对基座与 LoRA 永远来自同一个已加载模型，模型变更即同步换配对基座；
- LoRA **rank 16 / alpha 32**、lr **2e-4**、**3 epoch**、max_len **4096**；
- 精度自动检测：GPU 支持 bf16 用 **bf16**，否则用兼容 **fp16**；两者互斥、不硬编码。
  Qwen3.5 全程**非 4-bit** 训练（依赖安装按当前官方 Unsloth 指南）；
- 只对 assistant 段计损失（prompt/completion 格式，切分用模型自己的 chat template）；
- 训练前审计 token 长度并确认 **4096 不会截断任何 assistant 答案**；
- 导出：**基座 GGUF** 与 **微调 GGUF** 用同一种量化（默认 q4_k_m）、同一 tokenizer/
  chat template；另保存 **LoRA 适配器 + tokenizer + training_meta.json**，不只留 GGUF；
- 固定 seed=0，与全项目一致。系统提示词**不**进 Modelfile：由请求携带（与运行时一致）。

In [ ]:
# 按当前官方 Unsloth 指南安装：https://unsloth.ai/docs/get-started/install/pip-install
# （笔记本内需 % 前缀；官方推荐 pip install unsloth，uv 环境可加 --torch-backend=auto）
%pip install -q -U unsloth
%pip install -q -U trl datasets accelerate peft
# 装完如提示重启 runtime，先重启再从本格往下运行

In [ ]:
from pathlib import Path

# 基座候选：首选第一项；仅当某候选**确实不存在**（404 / Repo Not Found）才降级到下一个，
# 每次降级都会打印警告并记录进 training_meta.json（模型变更必须留痕）。
# 若你手动调整候选顺序，配对基座随之改变——最终以打印的 BASE_USED 为准。
BASE_CANDIDATES = [
    "unsloth/Qwen3.5-0.8B",
    "unsloth/Qwen3-1.7B",
    "unsloth/Qwen3-0.6B",
]
MAX_LEN       = 4096
LORA_R        = 16
LORA_ALPHA    = 32      # 任务书允许 16–32
LR            = 2e-4
EPOCHS        = 3       # 任务书 2–3；时间紧可改 2
BATCH_DEVICES = 2
GRAD_ACCUM    = 8       # 有效 batch = 16
SEED          = 0       # 全项目固定种子
USE_4BIT      = False   # 硬性要求：Qwen3.5 非 4-bit 训练
GGUF_QUANT    = "q4_k_m"  # 基座与微调模型用同一种量化；兼容问题换 q8_0（两者同步换）
VAL_N         = 8       # 训练集尾部留出的验证切片（不参与训练），供笔记本内冒烟
TRAIN_JSONL   = "train.jsonl"
# 注意：本笔记本绝不加载 test.jsonl（测试集隔离，见顶部说明）

In [ ]:
# 已手动上传可跳过本格
if not Path(TRAIN_JSONL).exists():
    try:
        from google.colab import files
        print("请选择本机的 ml/lora/datasets/train.jsonl")
        files.upload()
    except ImportError:
        print("非 Colab：把 train.jsonl 复制到当前目录")
assert Path(TRAIN_JSONL).exists(), "缺少 train.jsonl"
import hashlib, json
TRAIN_SHA256 = hashlib.sha256(Path(TRAIN_JSONL).read_bytes()).hexdigest()
rows = [json.loads(l) for l in open(TRAIN_JSONL, encoding="utf-8") if l.strip()]
assert all(len(r.get("messages", [])) == 3 for r in rows), "每行应为 system/user/assistant 三段"
systems = {r["messages"][0]["content"] for r in rows}
print(f"训练样本 {len(rows)} 条；system 提示词唯一: {len(systems) == 1}")
print(f"train.jsonl SHA256 = {TRAIN_SHA256}")

In [ ]:
import traceback
from unsloth import FastLanguageModel

MISSING_MARKERS = ("404", "not found", "does not exist", "repository not found",
                   "entry not found", "no such")

def _is_model_missing(exc: Exception) -> bool:
    text = f"{type(exc).__name__} {exc}".lower()
    return any(m in text for m in MISSING_MARKERS)

model = tokenizer = None
BASE_USED = None
FALLBACK_LOG = []  # 每次降级留痕：(候选, 失败原因)
for index, candidate in enumerate(BASE_CANDIDATES):
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=candidate,
            max_seq_length=MAX_LEN,
            dtype=None,            # 加载精度由 Unsloth 按硬件自动选择；训练精度在下方显式检测
            load_in_4bit=USE_4BIT,
        )
        BASE_USED = candidate
        break
    except Exception as exc:
        if _is_model_missing(exc) and index + 1 < len(BASE_CANDIDATES):
            reason = f"{type(exc).__name__}: {str(exc)[:160]}"
            FALLBACK_LOG.append({"rejected": candidate, "reason": reason})
            print("=" * 72)
            print(f"⚠ 模型变更：{candidate} 不存在，降级为 {BASE_CANDIDATES[index + 1]}")
            print(f"  原因: {reason}")
            print("  配对基座随之改变；该变更将写入 training_meta.json")
            print("=" * 72)
            continue
        print(f"[终止] {candidate} 加载失败，且不是「模型不存在」——不静默换模型：")
        traceback.print_exc()
        raise
assert model is not None, "候选基座全部不存在：到 Unsloth 文档核对模型名后更新 BASE_CANDIDATES"
MODEL_CHANGED = BASE_USED != BASE_CANDIDATES[0]
print(f"\n最终基座 BASE_USED = {BASE_USED}（是否发生模型变更: {MODEL_CHANGED}）")

In [ ]:
# 在挂载 LoRA 之前导出「未微调配对基座」GGUF——与微调后模型同量化、同 tokenizer/chat template。
# 模型若变更，这里导出的就是变更后的那一版（配对自动同步）。
import glob, shutil

model.save_pretrained_gguf("harborguard_base_gguf", tokenizer, quantization_method=GGUF_QUANT)
base_src = sorted(glob.glob("harborguard_base_gguf/**/*.gguf", recursive=True))[0]
shutil.copy(base_src, "harborguard-base.gguf")
print(f"配对基座 harborguard-base.gguf（{Path('harborguard-base.gguf').stat().st_size / 1e9:.2f} GB，{GGUF_QUANT}）")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
from datasets import Dataset

def to_prompt_completion(messages):
    """prompt = system+user（含生成提示符），completion = assistant 段（含结束符）。
    切分用模型自己的 chat template，保证与推理请求格式完全一致。"""
    full = tokenizer.apply_chat_template(messages, tokenize=False)
    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True)
    assert full.startswith(prompt), "chat template 切分失败：检查该基座的模板"
    return {"prompt": prompt, "completion": full[len(prompt):]}

# 训练集尾部留出 VAL_N 条做验证切片（不参与训练）；test.jsonl 不进本笔记本。
val_rows = rows[-VAL_N:]
train_rows = rows[:-VAL_N]

# ---- 训练前 token 长度审计：确认 MAX_LEN 不会截断 assistant 答案 ----
formatted = [to_prompt_completion(r["messages"]) for r in rows]
def _n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False)["input_ids"])
stats = {
    "prompt": [_n_tokens(f["prompt"]) for f in formatted],
    "completion": [_n_tokens(f["completion"]) for f in formatted],
}
totals = [p + c for p, c in zip(stats["prompt"], stats["completion"])]
over = [(i, t) for i, t in enumerate(totals) if t > MAX_LEN]
audit = {
    "max_prompt_tokens": max(stats["prompt"]),
    "max_completion_tokens": max(stats["completion"]),
    "max_total_tokens": max(totals),
    "mean_total_tokens": round(sum(totals) / len(totals), 1),
    "n_samples": len(totals),
    "n_over_limit": len(over),
    "max_len": MAX_LEN,
}
print("token 审计:", audit)
assert not over, (
    f"{len(over)} 条样本总长超过 MAX_LEN={MAX_LEN}，assistant 答案会被截断——"
    "请调大 MAX_LEN 后重跑（不要截断答案）"
)
print("✓ 全部样本总长 ≤ MAX_LEN，assistant 答案不会被截断")

train_ds = Dataset.from_list(formatted[:-VAL_N])
print(f"训练 {len(train_ds)} 条（留出 {len(val_rows)} 条验证切片）")
print("PROMPT 尾部:", formatted[0]["prompt"][-140:])
print("COMPLETION:", formatted[0]["completion"][:200])

In [ ]:
import inspect, torch
from trl import SFTConfig, SFTTrainer

# ---- 精度自动检测：支持 bf16 用 bf16，否则用兼容 fp16；互斥、不硬编码 ----
BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
PRECISION = "bf16" if BF16_OK else "fp16"
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '无'}；训练精度: {PRECISION}")

cfg = dict(
    output_dir="harborguard-lora-out",
    per_device_train_batch_size=BATCH_DEVICES,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    seed=SEED,
    packing=False,
    dataset_num_proc=2,
    report_to="none",
)
cfg["bf16"] = BF16_OK
cfg["fp16"] = not BF16_OK
assert cfg["bf16"] != cfg["fp16"], "bf16 与 fp16 不得同时开启"
# 兼容 TRL 版本差异：max_length（新）/ max_seq_length（旧）
params = inspect.signature(SFTConfig.__init__).parameters
cfg["max_length" if "max_length" in params else "max_seq_length"] = MAX_LEN

def build_trainer():
    tparams = inspect.signature(SFTTrainer.__init__).parameters
    key = "processing_class" if "processing_class" in tparams else "tokenizer"
    return SFTTrainer(model=model, train_dataset=train_ds, args=SFTConfig(**cfg), **{key: tokenizer})

trainer = build_trainer()
trainer.train()

In [ ]:
import time
FastLanguageModel.for_inference(model)

# 冒烟只用训练集尾部留出的验证切片；test.jsonl 不进本笔记本。
demo_messages = val_rows[0]["messages"]
demo_prompt = tokenizer.apply_chat_template(demo_messages[:-1], tokenize=False, add_generation_prompt=True)
inputs = tokenizer(demo_prompt, return_tensors="pt").to(model.device)

started = time.perf_counter()
output = model.generate(**inputs, max_new_tokens=400, do_sample=False)
latency = time.perf_counter() - started
answer = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Colab 生成耗时 {latency:.1f}s（验收：本机 ≤ 10s，此处仅作量级参考）\n")
print(answer)
try:
    parsed = json.loads(answer)
    print("\nJSON 解析: OK；字段:", sorted(parsed))
except Exception as exc:
    print("\nJSON 解析失败:", exc)

In [ ]:
from datetime import datetime, timezone
import glob, shutil


# ---- 保存 LoRA 适配器 + tokenizer + 训练配置（不只留 GGUF） ----
ADAPTER_DIR = "harborguard-lora-adapter"
model.save_pretrained(ADAPTER_DIR)          # PEFT 模型：只存适配器权重
tokenizer.save_pretrained(ADAPTER_DIR)

def _versions():
    import platform
    import trl, transformers, unsloth
    return {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "trl": getattr(trl, "__version__", "unknown"),
        "unsloth": getattr(unsloth, "__version__", "unknown"),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    }

def _base_revision():
    try:
        from huggingface_hub import model_info
        return model_info(BASE_USED).sha
    except Exception:
        return None  # 拿不到 revision 时如实留空，不编造

TRAINING_META = {
    "recorded_at": datetime.now(timezone.utc).isoformat(),
    "base_model_requested": BASE_CANDIDATES[0],
    "base_model_used": BASE_USED,
    "base_revision": _base_revision(),
    "model_changed": MODEL_CHANGED,
    "fallback_log": FALLBACK_LOG,
    "precision": PRECISION,
    "load_in_4bit": USE_4BIT,
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": 0.0,
             "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                                "gate_proj", "up_proj", "down_proj"]},
    "training": {"lr": LR, "epochs": EPOCHS, "batch_devices": BATCH_DEVICES,
                 "grad_accum": GRAD_ACCUM, "seed": SEED, "max_len": MAX_LEN},
    "token_audit": audit,
    "train_jsonl_sha256": TRAIN_SHA256,
    "train_samples": len(rows), "trained_samples": len(train_rows), "val_holdout": len(val_rows),
    "gguf_quant": GGUF_QUANT,
    "exports": {"lora_gguf": "harborguard.gguf", "paired_base_gguf": "harborguard-base.gguf",
               "adapter_dir": ADAPTER_DIR},
    "versions": _versions(),
}
Path("training_meta.json").write_text(
    json.dumps(TRAINING_META, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
Path(ADAPTER_DIR, "training_meta.json").write_text(
    json.dumps(TRAINING_META, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps({k: TRAINING_META[k] for k in
      ("base_model_used", "base_revision", "model_changed", "precision", "gguf_quant")},
      ensure_ascii=False, indent=2))

# ---- 导出微调后 GGUF（与配对基座同量化） ----
model.save_pretrained_gguf("harborguard_gguf", tokenizer, quantization_method=GGUF_QUANT)
lora_src = sorted(glob.glob("harborguard_gguf/**/*.gguf", recursive=True))[0]
shutil.copy(lora_src, "harborguard.gguf")
print(f"微调模型 harborguard.gguf（{Path('harborguard.gguf').stat().st_size / 1e9:.2f} GB，{GGUF_QUANT}）")
print("训练摘要已写入 training_meta.json（下载时一并带走）")

In [ ]:
try:
    from google.colab import files
    files.download("harborguard.gguf")
    files.download("harborguard-base.gguf")
    files.download("training_meta.json")
except ImportError:
    print("非 Colab：三个文件已在当前目录")

## 下载后的本机部署（MacBook / 演示机）

把两个 GGUF 放到仓库 `ml/lora/` 目录（与 `Modelfile`、`Modelfile.base` 同目录，GGUF 已被 gitignore）：

```bash
cd ml/lora
ollama create harborguard-lora -f Modelfile        # 微调模型
ollama create harborguard-base -f Modelfile.base   # 未微调配对基座（同量化/同模板/同参数）
ollama run harborguard-lora "你好"                  # 冒烟
python -m ml.lora.eval                             # 配对基座 vs LoRA + 系统基线 → metrics.json
curl -s localhost:8000/api/metrics                 # 后端读取 {yolo, lora}
```

`training_meta.json` 建议复制到 `ml/lora/`（已被 gitignore，不入库），记录实际使用的
基座 ID / revision / 精度 / 超参 / token 审计，供指标卡溯源。

### 排错 / 降级
- 基座候选不存在 → 笔记本会**明确警告并记录**后降级；其他异常直接终止，不会静默换模型；
- 显存不足 → `BATCH_DEVICES=1` 并把 `GRAD_ACCUM` 提到 16；时间不足 → `EPOCHS=2`；
- GGUF 导出或 Ollama 加载异常 → 基座与微调模型**同步**换 `q8_0` 重导（改 `GGUF_QUANT` 重跑导出格）；
- 训练失败时按任务书兜底：「基座模型 + 提示词」做本地路径（`OLLAMA_MODEL=qwen3:1.7b`），指标卡如实显示。